In [2]:
# 1) Imports & data load
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

# silence TF/Keras logs
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
warnings.filterwarnings("ignore")

# load your pre‐split datasets
import joblib
X_tr, y_tr = joblib.load("data/processed/train.pkl")
X_val, y_val = joblib.load("data/processed/val.pkl")
X_te, y_te = joblib.load("data/processed/test.pkl")

# fixed seeds & trials
SEEDS = [0, 42, 7, 123, 999]
N_TRIALS = 20

# unified search space for your cfg
cfg = {
    "mlp": {
        "search_space": {
            "n_layers":       (1, 3),       # trial.suggest_int
            "units":          (32, 256),    # trial.suggest_int
            "dropout":        (0.0, 0.5),   # trial.suggest_float
            "learning_rate":  (1e-4, 1e-2), # trial.suggest_float log
        },
        "max_epochs": 50,
        "patience":   10,
        "batch_size": 256
    },
    "general": {
        "n_trials": N_TRIALS
    }
}

In [4]:
# ────────────────────────────────────────────────────────
# 2) Optuna‐Tuned MLP
# ────────────────────────────────────────────────────────
import optuna
from tensorflow import keras
from tensorflow.keras import layers

def tune_optuna_mlp(seed):
    def objective(trial):
        n_layers = trial.suggest_int("n_layers", *cfg["mlp"]["search_space"]["n_layers"])
        units    = trial.suggest_int("units",    *cfg["mlp"]["search_space"]["units"])
        dropout  = trial.suggest_float("dropout", *cfg["mlp"]["search_space"]["dropout"])
        lr       = trial.suggest_float("learning_rate", 
                                       cfg["mlp"]["search_space"]["learning_rate"][0],
                                       cfg["mlp"]["search_space"]["learning_rate"][1],
                                       log=True)
        # build model
        model = keras.Sequential()
        model.add(layers.Input(shape=(X_tr.shape[1],)))
        for _ in range(n_layers):
            model.add(layers.Dense(units, activation="relu"))
            model.add(layers.Dropout(dropout))
        model.add(layers.Dense(1, activation="linear"))
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss="mse",
            metrics=[keras.metrics.RootMeanSquaredError(name="rmse")]
        )
        # fit with early stopping
        hist = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=cfg["mlp"]["max_epochs"],
            batch_size=cfg["mlp"]["batch_size"],
            callbacks=[keras.callbacks.EarlyStopping(
                monitor="val_rmse",
                patience=cfg["mlp"]["patience"],
                restore_best_weights=True,
                verbose=0
            )],
            verbose=0
        )
        return float(min(hist.history["val_rmse"]))
    
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=cfg["general"]["n_trials"])
    return study.best_params, study.best_value

In [6]:
# ────────────────────────────────────────────────────────
# 3) Hyperopt‐Tuned MLP
# ────────────────────────────────────────────────────────
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

def tune_hyperopt_mlp(seed):
    space = {
        "n_layers":      hp.quniform("n_layers", *cfg["mlp"]["search_space"]["n_layers"], 1),
        "units":         hp.quniform("units",    *cfg["mlp"]["search_space"]["units"],    1),
        "dropout":       hp.uniform("dropout",   *cfg["mlp"]["search_space"]["dropout"]),
        "learning_rate": hp.loguniform("learning_rate",
                                       np.log(cfg["mlp"]["search_space"]["learning_rate"][0]),
                                       np.log(cfg["mlp"]["search_space"]["learning_rate"][1])),
    }
    trials = Trials()
    
    def objective(params):
        n_layers = int(params["n_layers"])
        units    = int(params["units"])
        dropout  = params["dropout"]
        lr       = params["learning_rate"]
        # build & fit
        model = keras.Sequential()
        model.add(layers.Input(shape=(X_tr.shape[1],)))
        for _ in range(n_layers):
            model.add(layers.Dense(units, activation="relu"))
            model.add(layers.Dropout(dropout))
        model.add(layers.Dense(1, activation="linear"))
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss="mse",
            metrics=[keras.metrics.RootMeanSquaredError(name="rmse")]
        )
        hist = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=cfg["mlp"]["max_epochs"],
            batch_size=cfg["mlp"]["batch_size"],
            callbacks=[keras.callbacks.EarlyStopping(
                monitor="val_rmse",
                patience=cfg["mlp"]["patience"],
                restore_best_weights=True,
                verbose=0
            )],
            verbose=0
        )
        rmse = float(min(hist.history["val_rmse"]))
        return {"loss": rmse, "status": STATUS_OK}
    
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=cfg["general"]["n_trials"],
        trials=trials,
        rstate=np.random.default_rng(seed)
    )
    # extract losses
    best_rmse = np.min(trials.losses())
    return {
        "n_layers": int(best["n_layers"]),
        "units":    int(best["units"]),
        "dropout":  float(best["dropout"]),
        "learning_rate": float(best["learning_rate"])
    }, best_rmse

In [8]:
# ────────────────────────────────────────────────────────
# 4) Skopt‐Tuned MLP
# ────────────────────────────────────────────────────────
from skopt import gp_minimize
from skopt.space import Real, Integer

def tune_skopt_mlp(seed):
    dims = [
        Integer(*cfg["mlp"]["search_space"]["n_layers"], name="n_layers"),
        Integer(*cfg["mlp"]["search_space"]["units"],    name="units"),
        Real(*cfg["mlp"]["search_space"]["dropout"],     name="dropout"),
        Real(*cfg["mlp"]["search_space"]["learning_rate"], prior="log-uniform", name="learning_rate"),
    ]
    def objective(x):
        n_layers, units, dropout, lr = x
        model = keras.Sequential()
        model.add(layers.Input(shape=(X_tr.shape[1],)))
        for _ in range(n_layers):
            model.add(layers.Dense(units, activation="relu"))
            model.add(layers.Dropout(dropout))
        model.add(layers.Dense(1, activation="linear"))
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss="mse",
            metrics=[keras.metrics.RootMeanSquaredError(name="rmse")]
        )
        hist = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=cfg["mlp"]["max_epochs"],
            batch_size=cfg["mlp"]["batch_size"],
            callbacks=[keras.callbacks.EarlyStopping(
                monitor="val_rmse",
                patience=cfg["mlp"]["patience"],
                restore_best_weights=True,
                verbose=0
            )],
            verbose=0
        )
        return float(min(hist.history["val_rmse"]))
    
    res = gp_minimize(objective, dims,
                      n_calls=cfg["general"]["n_trials"],
                      random_state=seed)
    return {
        "n_layers":    int(res.x[0]),
        "units":       int(res.x[1]),
        "dropout":     float(res.x[2]),
        "learning_rate": float(res.x[3])
    }, res.fun

In [10]:
# ────────────────────────────────────────────────────────
# 5) Run all three and collect results
# ────────────────────────────────────────────────────────
records = []
for pkg, fn in [("Optuna", tune_optuna_mlp),
                ("Hyperopt", tune_hyperopt_mlp),
                ("Skopt", tune_skopt_mlp)]:
    for seed in SEEDS:
        params, val_rmse = fn(seed)
        records.append({
            "package":    pkg,
            "seed":       seed,
            "val_rmse":   round(val_rmse,3),
            **params
        })

df_tune = pd.DataFrame(records)
df_tune.to_csv("rq1_mlp_tuning.csv", index=False)
print("→ tuning results → rq1_mlp_tuning.csv")
display(df_tune)

[I 2025-06-10 23:10:55,153] A new study created in memory with name: no-name-3a1558c8-c668-492b-9be9-8f3a7b58b728
[I 2025-06-10 23:11:36,614] Trial 0 finished with value: 8652.3232421875 and parameters: {'n_layers': 2, 'units': 192, 'dropout': 0.30138168803582194, 'learning_rate': 0.0012296071107325709}. Best is trial 0 with value: 8652.3232421875.
[I 2025-06-10 23:12:17,748] Trial 1 finished with value: 7235.62451171875 and parameters: {'n_layers': 2, 'units': 177, 'dropout': 0.21879360563134626, 'learning_rate': 0.006074996073425695}. Best is trial 1 with value: 7235.62451171875.
[I 2025-06-10 23:13:00,497] Trial 2 finished with value: 7946.328125 and parameters: {'n_layers': 3, 'units': 118, 'dropout': 0.3958625190413323, 'learning_rate': 0.0011423254155608382}. Best is trial 1 with value: 7235.62451171875.
[I 2025-06-10 23:13:52,983] Trial 3 finished with value: 13470.580078125 and parameters: {'n_layers': 2, 'units': 240, 'dropout': 0.03551802909894347, 'learning_rate': 0.00014936

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [11:33<00:00, 34.70s/trial, best loss: 6907.97802734375]
→ tuning results → rq1_mlp_tuning.csv


,package,seed,val_rmse,n_layers,units,dropout,learning_rate
0,Optuna,0,7048.931,3,222,0.110057,0.004474
1,Optuna,42,6889.163,2,180,0.060731,0.005370
2,Optuna,7,6824.281,2,247,0.055759,0.008873
3,Optuna,123,6722.890,3,59,0.107799,0.004006
4,Optuna,999,6854.426,3,238,0.010338,0.008959
5,Hyperopt,0,7086.167,3,173,0.123629,0.002306
6,Hyperopt,42,7092.317,2,132,0.114918,0.008954
7,Hyperopt,7,7030.392,3,255,0.217407,0.005913
8,Hyperopt,123,6721.109,3,178,0.129730,0.009873
9,Hyperopt,999,6907.978,2,65,0.005635,0.008240


In [14]:
# ──────────────────────────────────────────────────────────────────────────────
# 7) Retrain best MLP on (train+val), evaluate on test & save rq1_mlp_final.csv
# ──────────────────────────────────────────────────────────────────────────────

# Pick the very best trial (min val_rmse)
best = df_tune.loc[df_tune["val_rmse"].idxmin()]

# Extract hyperparams
n_layers = int(best["n_layers"])
units    = int(best["units"])
lr        = float(best["learning_rate"])
dropout   = float(best["dropout"])

# Build model
tf.keras.backend.clear_session()
model = keras.Sequential()
model.add(keras.layers.Input(shape=(X_tr.shape[1],)))
for _ in range(n_layers):
    model.add(keras.layers.Dense(units, activation="relu"))
    model.add(keras.layers.Dropout(dropout))
model.add(keras.layers.Dense(1, activation="linear"))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr),
    loss="mse",
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)

# Retrain on combined train+val
X_full = np.vstack([X_tr, X_val])
y_full = np.concatenate([y_tr, y_val])

t0 = time.time()
model.fit(
    X_full, y_full,
    batch_size=256,
    epochs=50,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="rmse",
            patience=10,
            restore_best_weights=True,
            verbose=0
        )
    ],
    verbose=0
)
train_time = time.time() - t0

# Evaluate on test
t1 = time.time()
y_pred = model.predict(X_te, verbose=0).ravel()
pred_time = time.time() - t1

test_rmse = mean_squared_error(y_te, y_pred, squared=False)
test_r2   = r2_score(y_te, y_pred)

# Save summary
df_final = pd.DataFrame([{
    "n_layers":      n_layers,
    "units":         units,
    "learning_rate": lr,
    "dropout":       dropout,
    "train_time_s":  round(train_time, 3),
    "pred_time_s":   round(pred_time,  3),
    "test_rmse":     round(test_rmse,  3),
    "test_r2":       round(test_r2,     4),
}])
df_final.to_csv("rq1_mlp_final.csv", index=False)
print("→ final results saved to rq1_mlp_final.csv")
display(df_final)

→ final results saved to rq1_mlp_final.csv


,n_layers,units,learning_rate,dropout,train_time_s,pred_time_s,test_rmse,test_r2
0,3,178,0.009873,0.12973,39.77,0.586,6359.375,0.9281
